In [ ]:
# Notebook 03: Vector Store Setup
#
!pip install chromadb sentence-transformers -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.7/20.7 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.0/208.0 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 7.0 MB/s eta 

In [ ]:
from pathlib import Path

# Google Drive mount
from google.colab import drive
drive.mount('/content/drive')

# Base directory - all data saved here
BASE_DIR = Path("/content/drive/MyDrive/multi_agent_rag_project")
BASE_DIR.mkdir(parents=True, exist_ok=True)

# Now redefine all paths
DATA_DIR = BASE_DIR / "data/raw/arxiv_papers"
CHUNKS_DIR = BASE_DIR / "data/processed/chunks"
CHROMA_DIR = BASE_DIR / "data/vector_store"
MODEL_DIR = BASE_DIR / "models"
RESULTS_DIR = BASE_DIR / "results"

# Create all directories
for dir_path in [DATA_DIR, CHUNKS_DIR, CHROMA_DIR, MODEL_DIR, RESULTS_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)


Mounted at /content/drive


In [ ]:


import json
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
from pathlib import Path
import numpy as np

CHUNKS_FILE = BASE_DIR / "data/processed/chunks/all_chunks.json"
CHROMA_DIR = BASE_DIR / "data/vector_store"
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

print("Vector store setup initialized")

Vector store setup initialized


In [ ]:


# Load chunks
with open(CHUNKS_FILE, 'r') as f:
    chunks = json.load(f)

print(f"Loaded {len(chunks)} chunks")

Loaded 12728 chunks


In [ ]:


# Initialize embedding model
print("Loading embedding model...")
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print("Model loaded successfully")

Loading embedding model...
Model loaded successfully


In [ ]:


# Initialize ChromaDB
client = chromadb.PersistentClient(path=str(CHROMA_DIR))

# Create collection
collection = client.get_or_create_collection(
    name="arxiv_papers",
    metadata={"description": "ArXiv research papers for RAG"}
)

print(f"ChromaDB collection created: {collection.name}")
print(f"Existing documents: {collection.count()}")

ChromaDB collection created: arxiv_papers
Existing documents: 0


In [ ]:

# Batch embedding and insertion
batch_size = 100
total_batches = (len(chunks) + batch_size - 1) // batch_size

print(f"\nProcessing {len(chunks)} chunks in {total_batches} batches...")

for i in tqdm(range(0, len(chunks), batch_size), desc="Adding to ChromaDB"):
    batch = chunks[i:i+batch_size]

    # Prepare data
    texts = [c['text'] for c in batch]
    ids = [c['chunk_id'] for c in batch]
    metadatas = [
        {
            'arxiv_id': c['arxiv_id'],
            'title': c['title'],
            'authors': ', '.join(c['authors'][:3]),  # First 3 authors
            'published': c['published'],
            'categories': ', '.join(c['categories']),
            'chunk_index': c['chunk_index'],
        }
        for c in batch
    ]

    # Generate embeddings
    embeddings = model.encode(texts, show_progress_bar=False).tolist()

    # Add to collection
    collection.add(
        documents=texts,
        embeddings=embeddings,
        ids=ids,
        metadatas=metadatas
    )

print(f"\nTotal documents in collection: {collection.count()}")


Processing 12728 chunks in 128 batches...


Adding to ChromaDB: 100%|██████████| 128/128 [42:54<00:00, 20.11s/it]


Total documents in collection: 12728


In [ ]:

# Test retrieval
test_query = "What are transformers in deep learning?"
print(f"\nTest query: '{test_query}'")

query_embedding = model.encode([test_query])[0].tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)

print("\nTop 5 results:")
for i, (doc, metadata, distance) in enumerate(zip(
    results['documents'][0],
    results['metadatas'][0],
    results['distances'][0]
), 1):
    print(f"\n{i}. {metadata['title']}")
    print(f"   Distance: {distance:.4f}")
    print(f"   Preview: {doc[:150]}...")




Test query: 'What are transformers in deep learning?'

Top 5 results:

1. The Text Classification Pipeline: Starting Shallow going Deeper
   Distance: 0.7348
   Preview: Future advancements in this ﬁeld are expected to leverage the Transformer architecture (Vaswani et al., 2017), which has proven to be faster and more ...

2. A Novel Shape Guided Transformer Network for Instance Segmentation in Remote Sensing Images
   Distance: 0.7666
   Preview: This work was supported by the National Natural Science Foundation of China under Grant 42171430 and Grant 42030102. (Corresponding author: Shunping J...

3. JADE: Joint-aware Latent Diffusion for 3D Human Generative Modeling
   Distance: 0.8023
   Preview: Scalable diffusion models with transformers. In Proceedings of the IEEE/CVF Inter- national Conference on Computer Vision, pages 4195–4205, 2023. 2, 5...

4. Leaf diseases detection using deep learning methods
   Distance: 0.8037
   Preview: This ability to extract features makes deep lea